# Synthetic customer churn pilot

Controlled synthetic clients and local orders; no real customer data or network access.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier

## Simulate customers

In [ ]:
customers = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C04", "C05", "C06",
                    "C07", "C08", "C09", "C10", "C11", "C12"],
    "segment": ["small", "large", "small", "large", "small", "small",
                "small", "large", "large", "small", "small", "small"],
    "recency_days": [5, 10, 12, 8, 15, 20, 45, 50, 60, 75, 90, 55],
    "service_tickets": np.random.default_rng(7).integers(0, 5, size=12),
    "churned": [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
})
assert customers["customer_id"].is_unique

## Load local orders

In [ ]:
orders = pd.read_csv("orders.csv")

## Clean orders

In [ ]:
clean_orders = orders.drop_duplicates(subset=["order_id"], keep="first")
clean_orders = clean_orders.loc[
    (clean_orders["status"] == "completed") & clean_orders["units"].notna()
].copy()
clean_orders = clean_orders.assign(
    net_revenue=clean_orders["units"] * clean_orders["unit_price"]
    * (1 - clean_orders["discount"].fillna(0.0))
)
assert clean_orders["order_id"].is_unique

## Aggregate orders

In [ ]:
order_totals = clean_orders.groupby("customer_id", sort=True).agg(
    net_revenue=("net_revenue", "sum"),
    completed_orders=("order_id", "size"),
).reset_index()

## Join customer features

In [ ]:
features = customers.merge(
    order_totals, on="customer_id", how="left", validate="one_to_one"
)
features = features.fillna({"net_revenue": 0.0, "completed_orders": 0})
features = features.assign(completed_orders=features["completed_orders"].astype("int64"))
features = features.assign(
    average_order=features["net_revenue"].div(features["completed_orders"].replace(0, 1))
)
features = features.sort_values("customer_id").reset_index(drop=True)
assert features["customer_id"].is_unique

## Prepare features

In [ ]:
X = features[["recency_days", "net_revenue", "completed_orders", "average_order"]]
y = features["churned"]
holdout = features["customer_id"].isin(["C04", "C06", "C08", "C12"])

## Split fixed holdout

In [ ]:
X_train = X.loc[~holdout]
X_test = X.loc[holdout]
y_train = y.loc[~holdout]
y_test = y.loc[holdout]

## Train model

In [ ]:
model = DecisionTreeClassifier(max_depth=2, random_state=7)
model.fit(X_train, y_train)

## Predict holdout

In [ ]:
predictions = model.predict(X_test)

## Evaluate model

In [ ]:
accuracy = accuracy_score(y_test, predictions)
confusion = confusion_matrix(y_test, predictions, labels=[0, 1])
evaluation = features.loc[holdout, ["customer_id", "segment", "churned"]].assign(
    prediction=predictions
)
evaluation = evaluation.assign(
    correct=(evaluation["churned"] == evaluation["prediction"]).astype(int)
)
segment_metrics = evaluation.groupby("segment", sort=True).agg(
    support=("customer_id", "size"), accuracy=("correct", "mean")
).reset_index()
assert accuracy == 1.0
accuracy